# 🖼️ Notebook 1 — Vision Transformer (ViT) Training
## KYC Document Forgery Detection
**Project:** Multimodal Risk Assessment in FinTech Applications | Team 30

**Runtime:** GPU (T4) | **Est. Time:** 30–45 min

### Steps:
1. Mount Google Drive
2. Install dependencies
3. Download & prepare Aadhaar dataset (Roboflow)
4. Fine-tune ViT-B/16
5. Save `vit_model.pt` to Drive

In [ ]:
# ── STEP 0: Check GPU ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '❌ No GPU found! Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# ── STEP 1: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/MajorProject_Models'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ Drive mounted. Models will be saved to: {SAVE_DIR}')

In [ ]:
# ── STEP 2: Install Dependencies ────────────────────────────────────────────
!pip install -q timm roboflow Pillow torchvision scikit-learn matplotlib seaborn

In [ ]:
# ── STEP 3: Dataset Setup ────────────────────────────────────────────────────
# Option A: Use existing Aadhaar zip (upload from your PC)
# Upload 'Back Aadhaar Card.v2i.coco.zip' to Colab Files panel

# Option B: Create synthetic dataset (no upload needed)
import torch
import numpy as np
from PIL import Image, ImageDraw, ImageFilter
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import random, os

# Create synthetic authentic + tampered ID card images
def create_id_image(tampered=False):
    img = Image.new('RGB', (224, 224), color=(240, 230, 210))
    draw = ImageDraw.Draw(img)
    # Aadhaar-like layout
    draw.rectangle([10, 10, 214, 214], outline=(0,0,0), width=3)
    draw.rectangle([10, 10, 214, 50], fill=(255, 165, 0))
    draw.text((20, 20), 'GOVERNMENT OF INDIA', fill=(255,255,255))
    draw.text((20, 60), 'Name: Test User', fill=(0,0,0))
    draw.text((20, 85), 'DOB: 01/01/1990', fill=(0,0,0))
    draw.text((20, 110), 'XXXX XXXX 1234', fill=(0,0,0))
    draw.ellipse([160, 60, 210, 110], fill=(200,180,160))
    if tampered:
        # Add tamper noise: patch overlay + blurring
        for _ in range(random.randint(3, 8)):
            x, y = random.randint(10, 160), random.randint(55, 130)
            w, h = random.randint(20, 60), random.randint(10, 25)
            draw.rectangle([x, y, x+w, y+h], fill=(random.randint(200,255), random.randint(200,255), random.randint(200,255)))
            draw.text((x+2, y+2), 'XXXX', fill=(0,0,0))
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.5)))
    return img

# Generate dataset
DATA_DIR = '/content/kyc_dataset'
for split in ['train', 'val']:
    os.makedirs(f'{DATA_DIR}/{split}/authentic', exist_ok=True)
    os.makedirs(f'{DATA_DIR}/{split}/tampered', exist_ok=True)

# Train: 800 authentic + 800 tampered | Val: 100 + 100
for split, count in [('train', 800), ('val', 100)]:
    for i in range(count):
        create_id_image(tampered=False).save(f'{DATA_DIR}/{split}/authentic/{i}.jpg')
        create_id_image(tampered=True).save(f'{DATA_DIR}/{split}/tampered/{i}.jpg')
    print(f'✅ {split}: {count*2} images created')

print('\n📂 Dataset ready!')

In [ ]:
# ── STEP 4: Dataset Class ────────────────────────────────────────────────────
import torchvision

transform_train = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transform_val = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_ds = torchvision.datasets.ImageFolder(f'{DATA_DIR}/train', transform=transform_train)
val_ds   = torchvision.datasets.ImageFolder(f'{DATA_DIR}/val',   transform=transform_val)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)

print(f'✅ Classes: {train_ds.classes}')
print(f'   Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
# ── STEP 5: Load Pretrained ViT-B/16 ─────────────────────────────────────────
import timm
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Load ViT pretrained on ImageNet
model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=2)
model = model.to(device)

# Freeze all except head
for name, param in model.named_parameters():
    if 'head' not in name:
        param.requires_grad = False

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ ViT-B/16 loaded | Total params: {total:,} | Trainable: {trainable:,}')

In [ ]:
# ── STEP 6: Training ─────────────────────────────────────────────────────────
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=10)

EPOCHS = 10
best_val_acc = 0.0
history = {'train_loss': [], 'val_acc': []}

for epoch in range(EPOCHS):
    # Train
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    scheduler.step()

    # Validate
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    train_loss = running_loss / len(train_loader)
    history['train_loss'].append(train_loss)
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), f'{SAVE_DIR}/vit_model.pt')

    print(f'Epoch {epoch+1:02d}/{EPOCHS} | Loss: {train_loss:.4f} | Val Acc: {val_acc*100:.2f}% {"✅ Saved!" if val_acc == best_val_acc else ""}')

print(f'\n🎉 Training complete! Best Val Accuracy: {best_val_acc*100:.2f}%')
print(f'💾 Model saved: {SAVE_DIR}/vit_model.pt')

In [ ]:
# ── STEP 7: Plot Results ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], 'b-o'); ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch')
ax2.plot([a*100 for a in history['val_acc']], 'g-o'); ax2.set_title('Validation Accuracy (%)'); ax2.set_xlabel('Epoch')
plt.suptitle('ViT-B/16 — KYC Forgery Detection Training', fontsize=14)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/vit_training_plot.png', dpi=150)
plt.show()
print(f'📊 Plot saved to Drive!')

In [ ]:
# ── STEP 8: Download model to your PC ────────────────────────────────────────
from google.colab import files
print('⬇️ Downloading vit_model.pt to your PC...')
files.download(f'{SAVE_DIR}/vit_model.pt')
print('✅ Done! Place this file in: d:\\Major Project\\model\\vit_model.pt')